# Task 12.3 – American Options with Discrete Dividends

In [1]:
import numpy as np
import pandas as pd

In [2]:
def american_binomial_discrete_divs(S, K, T, r, sigma, option_type,
                                    div_times, div_amounts, N=500):
    """
    Price an American option with discrete cash dividends using the
    CRR binomial tree with the escrow (present-value) method.

    The stock price is decomposed as:
        S(t) = S*(t) + PV_remaining_dividends(t)
    where S* follows a GBM with no dividends.  The tree is built on S*,
    and the actual stock price at each node is S* + PV(remaining divs).

    Parameters
    ----------
    S : float          - Current stock price
    K : float          - Strike price
    T : float          - Time to maturity in years
    r : float          - Continuously compounded risk-free rate
    sigma : float      - Volatility of the dividend-adjusted stock process
    option_type : str  - 'Call' or 'Put'
    div_times : list   - Dividend payment times in years
    div_amounts : list - Dollar dividend amounts
    N : int            - Number of time steps

    Returns
    -------
    float : option price
    """
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    p = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)

    is_call = option_type.strip().lower() == 'call'

    # PV of all future dividends at time 0
    pv_all = sum(D * np.exp(-r * t) for D, t in zip(div_amounts, div_times))
    S_star = S - pv_all          # dividend-adjusted spot

    # Terminal layer: no remaining dividends at t = T
    j = np.arange(N + 1)
    S_star_T = S_star * u ** (N - j) * d ** j
    V = np.maximum(S_star_T - K, 0) if is_call else np.maximum(K - S_star_T, 0)

    # Backward induction
    for i in range(N - 1, -1, -1):
        t_i = i * dt
        # PV of dividends that have NOT yet been paid at node time t_i
        pv_rem = sum(D * np.exp(-r * (t_d - t_i))
                     for D, t_d in zip(div_amounts, div_times)
                     if t_d > t_i)

        j = np.arange(i + 1)
        S_star_i = S_star * u ** (i - j) * d ** j
        S_i = S_star_i + pv_rem          # actual stock price at each node

        # Roll back one step
        V = disc * (p * V[:i + 1] + (1 - p) * V[1:i + 2])
        intrinsic = np.maximum(S_i - K, 0) if is_call else np.maximum(K - S_i, 0)
        V = np.maximum(V, intrinsic)

    return V[0]

In [3]:
def parse_list(s):
    """Parse a comma-separated string into a list of floats."""
    return [float(x.strip()) for x in str(s).split(",")]


def process_discrete_divs(input_path, output_path, N=500):
    """
    Read option parameters (including discrete dividends) from input_path,
    compute American option price, print results, and save to output_path.
    """
    df = pd.read_csv(input_path).dropna(subset=["ID"])
    df["ID"] = df["ID"].astype(int)

    results = []
    for _, row in df.iterrows():
        T = row["DaysToMaturity"] / row["DayPerYear"]
        div_days   = parse_list(row["DividendDates"])
        div_amts   = parse_list(row["DividendAmts"])
        div_times  = [d / row["DayPerYear"] for d in div_days]   # convert to years

        price = american_binomial_discrete_divs(
            S=row["Underlying"],
            K=row["Strike"],
            T=T,
            r=row["RiskFreeRate"],
            sigma=row["ImpliedVol"],
            option_type=row["Option Type"],
            div_times=div_times,
            div_amounts=div_amts,
            N=N,
        )
        results.append({"ID": int(row["ID"]), "Value": price})

    out = pd.DataFrame(results, columns=["ID", "Value"])
    print(out.to_string(index=False))
    out.to_csv(output_path, index=False)
    print(f"\nSaved to {output_path}")
    return out

results_12_3 = process_discrete_divs("test12_3.csv", "testout_12.3.csv", N=500)

 ID     Value
  1 14.502227
  2 11.778420

Saved to testout_12.3.csv
